In [1]:
import os, random, time
import numpy as np
from PIL import Image
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision.models import resnet18


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)


Device: cuda


In [3]:
DATA_ROOT = "/kaggle/input"


RVF_ROOT = os.path.join(DATA_ROOT, "140k-real-and-fake-faces", "real_vs_fake", "real-vs-fake")


FFHQ_ROOT = os.path.join(DATA_ROOT, "ffhq-1024x1024", "images1024x1024")


DEEP_ROOT = os.path.join(DATA_ROOT, "deepdetect-2025", "ddata")


In [4]:
def assert_exists(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing path: {p}")
    return True

assert_exists(RVF_ROOT)
assert_exists(FFHQ_ROOT)
assert_exists(DEEP_ROOT)
print("✅ Roots found")


✅ Roots found


In [5]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")

def list_images(folder: str) -> List[str]:
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(IMG_EXTS):
                out.append(os.path.join(root, f))
    return out

def split_list(items: List[str], valid_ratio=0.1, seed=42):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)
    cut = int(len(items) * (1 - valid_ratio))
    train_idx, valid_idx = idx[:cut], idx[cut:]
    train_items = [items[i] for i in train_idx]
    valid_items = [items[i] for i in valid_idx]
    return train_items, valid_items


In [6]:
def get_rvf_split(split_name: str):
    real_dir = os.path.join(RVF_ROOT, split_name, "real")
    fake_dir = os.path.join(RVF_ROOT, split_name, "fake")
    assert_exists(real_dir); assert_exists(fake_dir)
    return list_images(real_dir), list_images(fake_dir)

rvf_train_real, rvf_train_fake = get_rvf_split("train")
rvf_valid_real, rvf_valid_fake = get_rvf_split("valid")

print("RVF train real/fake:", len(rvf_train_real), len(rvf_train_fake))
print("RVF valid real/fake:", len(rvf_valid_real), len(rvf_valid_fake))


RVF train real/fake: 50000 50000
RVF valid real/fake: 10000 10000


In [7]:
deep_fake_train_dir = os.path.join(DEEP_ROOT, "train", "fake")
deep_fake_valid_dir = os.path.join(DEEP_ROOT, "test", "fake")  

assert_exists(deep_fake_train_dir)
assert_exists(deep_fake_valid_dir)

deep_fake_train = list_images(deep_fake_train_dir)
deep_fake_valid = list_images(deep_fake_valid_dir)

print("DeepDetect fake train:", len(deep_fake_train))
print("DeepDetect fake valid (from test):", len(deep_fake_valid))


DeepDetect fake train: 41594
DeepDetect fake valid (from test): 10399


In [8]:
ffhq_all = list_images(FFHQ_ROOT)
print("FFHQ 1024 total:", len(ffhq_all))

ffhq_train, ffhq_valid = split_list(ffhq_all, valid_ratio=0.1, seed=42)
print("FFHQ train/valid:", len(ffhq_train), len(ffhq_valid))


FFHQ 1024 total: 70000
FFHQ train/valid: 63000 7000


In [9]:
def rgb_to_luma(img: Image.Image):
    arr = np.asarray(img).astype(np.float32) / 255.0
    return 0.299*arr[...,0] + 0.587*arr[...,1] + 0.114*arr[...,2]


In [10]:
def fft_mag(y):
    f = np.fft.fftshift(np.fft.fft2(y))
    return np.log1p(np.abs(f)).astype(np.float32)


In [11]:
def ring_normalize(mag, bins=64):
    h, w = mag.shape
    cy, cx = h//2, w//2
    yy, xx = np.ogrid[:h, :w]
    rr = np.sqrt((yy-cy)**2 + (xx-cx)**2)

    out = mag.copy()
    rmax = rr.max() + 1e-6
    edges = np.linspace(0, rmax, bins+1)

    for i in range(bins):
        mask = (rr >= edges[i]) & (rr < edges[i+1])
        if mask.any():
            vals = mag[mask]
            out[mask] = (vals - vals.mean()) / (vals.std() + 1e-6)

    return np.clip(out, -5, 5)


In [12]:
def compute_raps_map(mag, bins=64):
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)

    rmax = rr.max() + 1e-6
    edges = np.linspace(0, rmax, bins + 1)


    raps = np.zeros(bins, dtype=np.float32)
    for i in range(bins):
        mask = (rr >= edges[i]) & (rr < edges[i + 1])
        if mask.any():
            raps[i] = mag[mask].mean()


    raps = (raps - raps.mean()) / (raps.std() + 1e-6)


    raps_full = np.zeros_like(mag, dtype=np.float32)
    for i in range(bins):
        mask = (rr >= edges[i]) & (rr < edges[i + 1])
        raps_full[mask] = raps[i]

    return raps_full


In [13]:
def high_freq_residual(mag, cutoff=0.65):
    h, w = mag.shape
    cy, cx = h//2, w//2
    yy, xx = np.ogrid[:h, :w]
    rr = np.sqrt((yy-cy)**2 + (xx-cx)**2)
    mask = rr > cutoff * rr.max()
    out = np.zeros_like(mag)
    out[mask] = mag[mask]
    return (out - out.mean()) / (out.std() + 1e-6)


In [15]:
def center_crop(img, size):
    w, h = img.size
    if w < size or h < size:
        scale = size / min(w,h)
        img = img.resize((int(w*scale)+1, int(h*scale)+1))
    left = (img.width-size)//2
    top  = (img.height-size)//2
    return img.crop((left, top, left+size, top+size))


In [16]:
def build_freq_tensor(img, size=224):
    img = center_crop(img, size).resize((size, size))
    y = rgb_to_luma(img)

    mag = fft_mag(y)

    ch1 = ring_normalize(mag)                  
    ch2 = compute_raps_map(mag)                
    ch3 = high_freq_residual(mag)              

    x = np.stack([ch1, ch2, ch3], axis=0)
    return torch.tensor(x, dtype=torch.float32)


In [18]:
class FreqDataset(Dataset):
    def __init__(self, samples, train=True):
        self.samples = samples
        self.train = train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except:
            return self.__getitem__(random.randint(0,len(self.samples)-1))

        x = build_freq_tensor(img)
        return x, torch.tensor(label, dtype=torch.long)


In [19]:
class FreqNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet18(weights=None)
        base.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        base.fc = nn.Identity()
        self.backbone = base
        self.head = nn.Linear(512, 2)

    def forward(self, x):
        f = self.backbone(x)
        return self.head(f)


In [20]:
import os
print(os.listdir("/kaggle/input/datasets/kashirhanif"))

['frequency-model-checkpoint']


In [30]:
checkpoint_path = "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/freq_v5_epoch2_checkpoint.pt"


model = FreqNet().to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)


checkpoint = torch.load(checkpoint_path, map_location=DEVICE)


model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
start_epoch = checkpoint["epoch"]
best_acc = checkpoint["best_acc"]

print(f"✅ Resumed from epoch {start_epoch}")
print(f"✅ Best accuracy so far: {best_acc:.4f}")

✅ Resumed from epoch 4
✅ Best accuracy so far: 0.8519


In [38]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

best_acc = 0.8212   
start_epoch = 2     

In [23]:
DOMAINS = {
    "real_old": 0,
    "real_hq": 1,
    "fake_gan": 2,
    "fake_modern": 3,
}
ID2DOMAIN = {v:k for k,v in DOMAINS.items()}


In [24]:
def take_n(items: List[str], n: int, seed=42) -> List[str]:
    rng = np.random.RandomState(seed)
    if len(items) <= n:
        return items
    idx = rng.choice(len(items), size=n, replace=False)
    return [items[i] for i in idx]


N_REAL_OLD  = 50000   
N_REAL_HQ   = 50000   
N_FAKE_GAN  = 50000   
N_FAKE_MOD  = min(50000, len(deep_fake_train))  

train_real_old = take_n(rvf_train_real, N_REAL_OLD, seed=1)
train_real_hq  = take_n(ffhq_train,    N_REAL_HQ,  seed=2)
train_fake_gan = take_n(rvf_train_fake, N_FAKE_GAN, seed=3)
train_fake_mod = take_n(deep_fake_train, N_FAKE_MOD, seed=4)


V_REAL_OLD  = min(10000, len(rvf_valid_real))
V_REAL_HQ   = min(10000, len(ffhq_valid))
V_FAKE_GAN  = min(10000, len(rvf_valid_fake))
V_FAKE_MOD  = min(10000, len(deep_fake_valid))

valid_real_old = take_n(rvf_valid_real, V_REAL_OLD, seed=11)
valid_real_hq  = take_n(ffhq_valid,     V_REAL_HQ,  seed=12)
valid_fake_gan = take_n(rvf_valid_fake, V_FAKE_GAN, seed=13)
valid_fake_mod = take_n(deep_fake_valid, V_FAKE_MOD, seed=14)


train_samples_dom = []
train_samples_dom += [(p, 0, DOMAINS["real_old"])     for p in train_real_old]
train_samples_dom += [(p, 0, DOMAINS["real_hq"])      for p in train_real_hq]
train_samples_dom += [(p, 1, DOMAINS["fake_gan"])     for p in train_fake_gan]
train_samples_dom += [(p, 1, DOMAINS["fake_modern"])  for p in train_fake_mod]

valid_samples_dom = []
valid_samples_dom += [(p, 0, DOMAINS["real_old"])     for p in valid_real_old]
valid_samples_dom += [(p, 0, DOMAINS["real_hq"])      for p in valid_real_hq]
valid_samples_dom += [(p, 1, DOMAINS["fake_gan"])     for p in valid_fake_gan]
valid_samples_dom += [(p, 1, DOMAINS["fake_modern"])  for p in valid_fake_mod]


def count_dom(samples_dom, dom_id):
    return sum(1 for _,_,d in samples_dom if d == dom_id)

print("Train domain counts:",
      "real_old=",    count_dom(train_samples_dom, DOMAINS["real_old"]),
      "real_hq=",     count_dom(train_samples_dom, DOMAINS["real_hq"]),
      "fake_gan=",    count_dom(train_samples_dom, DOMAINS["fake_gan"]),
      "fake_modern=", count_dom(train_samples_dom, DOMAINS["fake_modern"]))

print("Valid domain counts:",
      "real_old=",    count_dom(valid_samples_dom, DOMAINS["real_old"]),
      "real_hq=",     count_dom(valid_samples_dom, DOMAINS["real_hq"]),
      "fake_gan=",    count_dom(valid_samples_dom, DOMAINS["fake_gan"]),
      "fake_modern=", count_dom(valid_samples_dom, DOMAINS["fake_modern"]))


train_samples = [(p, y) for (p, y, _) in train_samples_dom]
valid_samples = [(p, y) for (p, y, _) in valid_samples_dom]

print("Train sample example:", train_samples[0], "len=", len(train_samples[0]))
print("Valid sample example:", valid_samples[0], "len=", len(valid_samples[0]))


Train domain counts: real_old= 50000 real_hq= 50000 fake_gan= 50000 fake_modern= 41594
Valid domain counts: real_old= 10000 real_hq= 7000 fake_gan= 10000 fake_modern= 10000
Train sample example: ('/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real/64601.jpg', 0) len= 2
Valid sample example: ('/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid/real/16916.jpg', 0) len= 2


In [25]:
class FreqDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            return self.__getitem__(random.randint(0, len(self.samples)-1))

        x = build_freq_tensor(img, size=224)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


In [26]:

BATCH_SIZE = 64
NUM_WORKERS = 0   

train_ds = FreqDataset(train_samples)
valid_ds = FreqDataset(valid_samples)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("✅ loaders ready | train batches:", len(train_loader), "| valid batches:", len(valid_loader))


✅ loaders ready | train batches: 2993 | valid batches: 579


In [27]:
from tqdm import tqdm

def train_epoch(model, loader):
    model.train()
    total = 0
    loss_sum = 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        total += x.size(0)

        pbar.set_postfix(loss=loss.item())

    return loss_sum / total


In [28]:
@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    correct, total = 0, 0

    for x, y in tqdm(loader, desc="Validating", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return correct / total


In [ ]:
EPOCHS = 7  

for e in range(start_epoch + 1, EPOCHS + 1):
    print(f"\nEpoch {e}/{EPOCHS}")
    t0 = time.time()

    loss = train_epoch(model, train_loader)
    acc = eval_epoch(model, valid_loader)

    elapsed = (time.time() - t0) / 60
    print(f"Epoch {e} done | loss={loss:.4f} | acc={acc:.4f} | time={elapsed:.1f} min")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_freq_v5.pt")
        print("✅ Saved new best model")


Epoch 5/7


Epoch 5 done | loss=0.3133 | acc=0.8345 | time=144.3 min

Epoch 6/7


Epoch 6 done | loss=0.2667 | acc=0.6968 | time=142.5 min

Epoch 7/7


Validating:  43%|████▎     | 251/579 [11:22<24:57,  4.56s/it]                

In [41]:
checkpoint = {
    "epoch": 4,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "best_acc": best_acc,
}

torch.save(checkpoint, "freq_v5_epoch2_checkpoint.pt")
print("✅ Full checkpoint saved")

✅ Full checkpoint saved
